# Lab 2E: RAG Pipeline in Python

**Time**: ~20 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will build a Retrieval-Augmented Generation (RAG) pipeline using Cosmos DB vector search and Azure OpenAI.

## Prerequisites

- Python 3.10+ with `azure-cosmos`, `azure-identity`, `openai`, `numpy`, and `python-dotenv` installed: `pip install azure-cosmos azure-identity openai numpy python-dotenv`
- An active Azure CLI session. In PowerShell 7, run `az login`
- `COSMOS_ENDPOINT` environment variable set to your Cosmos DB account endpoint
- `FOUNDRY_ENDPOINT` environment variable set to your Foundry endpoint for chat completions
- `EMBEDDINGS_ENDPOINT` environment variable set to your Foundry endpoint for embeddings
- `COMPLETIONS_MODEL` environment variable set to the name of the Foundry model for chat completions
- `EMBEDDINGS_MODEL` environment variable set to the name of the Foundry model for embeddings

> **Lab order**: This lab seeds the shared RAG corpus into `WorkshopData/Docs` (partition key `rag`) that **Lab 2F** reads for its evaluation. Complete the labs in order - running them out of sequence can leave the `rag` partition empty or mixed with another lab's documents (Lab 4A also writes to this partition).

> **Pre-provisioned container**: The `Docs` container is created in advance by the workshop Bicep template with a vector embedding policy (`/embedding`, 1536 dimensions, cosine) and a DiskANN vector index. The lab stores and queries vectors without creating or configuring the container (Cosmos DB AAD tokens only authorize data-plane operations).

Run each cell in order to complete the steps.

In [ ]:
%pip install azure-cosmos azure-identity openai python-dotenv numpy --quiet

## Step 0: Initialize Connection

Set up the Cosmos DB and Azure OpenAI client connections.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
FOUNDRY_ENDPOINT = os.environ.get("FOUNDRY_ENDPOINT")
EMBEDDINGS_ENDPOINT = os.environ.get("EMBEDDINGS_ENDPOINT")
DB_NAME = "WorkshopData"
CONT_NAME = "Docs"
COMPLETIONS_MODEL = os.environ.get("COMPLETIONS_MODEL", "gpt41")
EMBEDDINGS_MODEL = os.environ.get("EMBEDDINGS_MODEL", "textembedding3small")

for var in ["COSMOS_ENDPOINT", "FOUNDRY_ENDPOINT", "EMBEDDINGS_ENDPOINT"]:
    if not os.environ.get(var):
        raise RuntimeError(f"{var} environment variable is required.")

print(f"Cosmos Endpoint:     {ENDPOINT}")
print(f"Foundry Endpoint:    {FOUNDRY_ENDPOINT}")
print(f"Embeddings Endpoint: {EMBEDDINGS_ENDPOINT}")
print(f"Database:            {DB_NAME}")
print(f"Container:           {CONT_NAME}")
print(f"Completions Model:   {COMPLETIONS_MODEL}")
print(f"Embeddings Model:    {EMBEDDINGS_MODEL}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import AzureCliCredential, get_bearer_token_provider
from openai import OpenAI

# Cosmos DB client (Entra ID auth)
cred = AzureCliCredential()
cosmos_client = CosmosClient(url=ENDPOINT, credential=cred)
db = cosmos_client.get_database_client(DB_NAME)
container = db.get_container_client(CONT_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}/{CONT_NAME}")

# Chat completions: Foundry endpoint, Entra ID auth.
foundry_token_provider = get_bearer_token_provider(cred, "https://ai.azure.com/.default")
foundry_client = OpenAI(
    base_url=f"{FOUNDRY_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=foundry_token_provider,
)

# Embeddings: Cognitive Services endpoint, Entra ID auth.
embeddings_token_provider = get_bearer_token_provider(cred, "https://cognitiveservices.azure.com/.default")
embeddings_client = OpenAI(
    base_url=f"{EMBEDDINGS_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=embeddings_token_provider,
)
print("Foundry chat client + embeddings client initialized")

## Step 1: Text Chunking and Seed Documents (Prebuilt)

Loads sample documents and chunks them into 512-character segments.

In [ ]:
def chunk_text(text: str, chunk_size: int = 512) -> list[str]:
    sentences = text.split(". ")
    chunks = []
    current_chunk = []
    current_size = 0
    
    for sentence in sentences:
        if current_size + len(sentence) > chunk_size and current_chunk:
            chunks.append(". ".join(current_chunk) + ".")
            current_chunk = [sentence]
            current_size = len(sentence)
        else:
            current_chunk.append(sentence)
            current_size += len(sentence)
    
    if current_chunk:
        chunks.append(". ".join(current_chunk) + ".")
    
    return chunks


sample_docs = [
    {"id": "doc1", "title": "Cosmos DB Overview", "content": "Azure Cosmos DB is a globally distributed, multi-model database service. It supports multiple API modes including SQL, MongoDB, Cassandra, Table, and Gremlin. Cosmos DB provides five consistency levels: Strong, Bounded Staleness, Session, Consistent Prefix, and Eventual."},
    {"id": "doc2", "title": "Vector Search", "content": "Azure Cosmos DB supports vector indexing for semantic similarity search queries. Vector search enables finding semantically similar items by comparing embeddings. The VectorDistance function calculates similarity scores between vectors."},
    {"id": "doc3", "title": "Data Modeling", "content": "Effective data modeling in Cosmos DB involves choosing the right partition key. Composite partition keys can help distribute load across partitions. Denormalization and fan-out patterns can optimize read performance."}
]

for doc in sample_docs:
    chunks = chunk_text(doc["content"])
    print(f"  Chunked '{doc['title']}': {len(chunks)} chunks")

print(f"Total: {len(sample_docs)} documents loaded")

## Step 2: Embed and Store Chunks (Prebuilt)

Generates embeddings for each chunk and upserts them to the `Docs` container with `partitionKey="rag"`. Each upsert prints its RU charge so you can see the cost of indexing the corpus before retrieval starts.

In [ ]:
def get_embedding(text: str) -> list[float]:
    resp = embeddings_client.embeddings.create(input=text, model=EMBEDDINGS_MODEL)
    return resp.data[0].embedding


for doc in sample_docs:
    chunks = chunk_text(doc["content"])

    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk)

        stored_doc = {
            "id": f"{doc['id']}_chunk_{i}",
            "title": doc["title"],
            "text": chunk,
            "embedding": embedding,
            "source": doc["id"],
            "partitionKey": "rag"
        }

        try:
            container.upsert_item(body=stored_doc)
            ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])
            print(f"  Stored chunk {i+1} of {doc['title']} (chunks={len(chunks)})")
            print(f"  RU charged: {ru}")
        except Exception as ex:
            print(f"  Error storing chunk: {ex}")

print("Chunk embedding and storage complete.")

## Step 3: RAG Retrieval

This is the retrieval half of RAG. The helper `retrieve_relevant` already embeds the user query, runs the iterator, and gathers results — your job is to write the `VectorDistance` query.

Replace the placeholder `vector_query` in the code cell with:

```python
vector_query = """
SELECT TOP @topk c.text, c.title, VectorDistance(c.embedding, @emb) AS score
FROM c
WHERE c.partitionKey = 'rag'
ORDER BY VectorDistance(c.embedding, @emb)
"""
```

**Expected output**: 3 retrieved chunks, ordered most-similar first, with scores.

In [ ]:
def retrieve_relevant(text_query: str, top_k: int = 3) -> list[dict]:
    query_embedding = get_embedding(text_query)

    vector_query = """
    SELECT TOP @topk c.text, c.title, VectorDistance(c.embedding, @emb) AS score
    FROM c
    WHERE c.partitionKey = 'rag'
    ORDER BY VectorDistance(c.embedding, @emb)
    """

    return list(container.query_items(
        query=vector_query,
        parameters=[
            {"name": "@topk", "value": top_k},
            {"name": "@emb", "value": query_embedding}
        ],
        partition_key="rag"
    ))


test_query = "What is vector search in Azure Cosmos DB?"
print(f"Retrieving for: \"{test_query}\"\n")

results = retrieve_relevant(test_query, 3)
print(f"Found {len(results)} results:\n")
for r in results:
    text = r.get("text", "")
    print(f"  Title: {r.get('title')}")
    print(f"  Score: {r.get('score')}")
    print(f"  Text:  {text[:100]}...\n")

## Step 4: RAG Generation

This is the generation half of RAG. The helper `generate_response` already calls `retrieve_relevant`, joins chunks into `context`, and sends the chat call. Your job is to write the **system prompt** that grounds the model in that context.

Replace the placeholder `system_prompt` in the code cell with:

```python
system_prompt = f"You are a helpful assistant. Answer the user's question based on the following context:\n\n<context>{context}</context>"
```

**Expected output**: An answer that draws from the retrieved chunks. The placeholder prompt has no `<context>` block, so the model answers from training data only.

In [ ]:
def generate_response(question: str) -> str:
    results = retrieve_relevant(question, 3)
    context = "\n\n".join(r.get("text", "") for r in results)

    system_prompt = f"You are a helpful assistant. Answer the user's question based on the following context:\n\n<context>{context}</context>"

    completion = foundry_client.chat.completions.create(
        model=COMPLETIONS_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        max_completion_tokens=500
    )
    return completion.choices[0].message.content


test_question = "What is vector search in Azure Cosmos DB?"
print(f"Question: {test_question}")
answer = generate_response(test_question)
print(f"\nAnswer:\n{answer}")